# Train path classification model

In [ ]:
from utils.device import get_device

device = get_device()

In [ ]:
import os
from image_segmentation.data import ImageDataset
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["FIVES"]

train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir

dataset = ImageDataset(data_dir=data_dir)
distances_hparams = dataset.get_dataset_distance_hparams()

## Overview of the training pipeline and it's main modules

![train_pipeline](../images/train_pipeline.png "Train pipeline diagram")

## Features Generator / Features extractor
The features generator is here a pretrained UNet (see [U-Net pretraining notebook (2)](./02_pretrain_unet.ipynb))

We load the checkpoint of the pretrained UNet and use it as a features generator for our path classification model.

We replace the last 32 to 1 layers convolutional layer of the pretrained UNet by a new one with 32 output channels, and we keep the pretrained weights for the rest of the UNet. We also set `freeze_pretrained` to False to allow fine-tuning of the pretrained UNet during the training of the path classification model.

In [ ]:
from path_neural_networks.models.features_generators import FeaturesGenerator, PretrainedUnetFeaturesGenerator
from utils.other import pretty_dict_print

# Switch between using the provided pretrained UNet or the one we pretrained ourselves
use_provided_unet = False

unet_ckpt_path = dataset_choice.get_first_checkpoint_path("unet_pretrained" if use_provided_unet else "unet_pretraining")
print("Using UNet checkpoint:", unet_ckpt_path)

features_generator: FeaturesGenerator = PretrainedUnetFeaturesGenerator(
    ckpt_path=unet_ckpt_path,
    device=device,
    out_channels=32,
    freeze_pretrained=False,
    skip_connection=False
)
features_generator_cfg = features_generator.as_dict()
pretty_dict_print(features_generator_cfg, init_message="Features generator configuration:")

## Path Sampler
For each path if the image, the path sampler will sampler a set squares patches of different sizes, for each coordinate in the path, and aggregate the features in these patches using the specified method (e.g. max pooling).

Using multiple square sizes allows to capture features at different scales, which can be beneficial for vessel segmentation where vessels can have varying widths, and because our vessels are note perfectly centered on the euclidean minimum path, so we want to capture features in a larger area around the path coordinates.

The aggregation method allows to summarize the features in the sampled patches into a single feature vector for each path coordinate, which can then be used as input to the path neural network.

At the end, we get a tensor of shape (n_features * n_scales, path_length) for each path, where n_features is the number of output channels of the features generator, n_scales is the number of different square sizes used for sampling, and path_length is the number of coordinates in the path.

![path_sampler](../images/path_features_sampling.png)

In [ ]:
from path_neural_networks.models.path_samplers import *

sampling_square_sizes = distances_hparams["scales_sampling_sizes"]
in_channels = features_generator.out_channels
sampling_aggregation_method = SamplingMaxAggregation()

path_sampler: PathSampler = MultiScaleSquarePathSampling(
    in_channels=in_channels,
    square_sizes=sampling_square_sizes,
    aggregation=sampling_aggregation_method
)
path_sampler_cfg = path_sampler.as_dict()
pretty_dict_print(path_sampler_cfg, init_message="Path sampler configuration:")

## Path encoder

The path encoder is a convolutional neural network that takes as input the features sampled along the path by the path sampler, and encodes them into a fixed-size feature vector that can be used for classification.

It is composed of a series of convolutional layers, followed by a global pooling operation (here a max pooling) to aggregate the features along the path, and optionally skip connections and residual blocks to improve the flow of information and gradients through the network.

At the end, we get a feature vector of size 256 for each path, which can then be used as input to a classifier to predict the class of the path (e.g. true vessel or false positive).

![path_encoder](../images/path_encoder.png "Path encoder diagram")

In [ ]:
from path_neural_networks.models.path_encoders import PathEncoder, ConvMaxPoolingPathEncoder

conv_path_residual_blocks = False
conv_path_skip_connections = False
conv_path_layers = [None, None, None]

path_encoder: PathEncoder = ConvMaxPoolingPathEncoder(
    in_channels=path_sampler.out_channels, 
    hidden_layers=conv_path_layers, 
    skip_connection=conv_path_skip_connections, 
    residual_blocks=conv_path_residual_blocks
)
path_encoder_cfg = path_encoder.as_dict()
pretty_dict_print(path_encoder_cfg, init_message="Path encoder configuration:")

## Path classifier

The path classifier is a fully connected network that takes as input the feature vector produced by the path encoder for each path, and outputs a binary classification (e.g. true vessel or false positive).

It is composed of a series of fully connected layers, optionally with dropout for regularization, ReLU activations, and layer normalization to improve training stability and performance.

The number of hidden layers and their sizes can be tuned to find the best architecture for the task at hand.

In [ ]:
from path_neural_networks.models.path_classifiers import PathClassifier, FCNPathClassifier

path_classifier_n_hidden_layers = 3
path_classifier_dropout = 0

path_classifier: PathClassifier = FCNPathClassifier(
    in_channels=path_encoder.out_channels,
    n_hidden_layers=path_classifier_n_hidden_layers,
    num_classes=1,
    dropout=path_classifier_dropout
)
path_classifier_cfg = path_classifier.as_dict()
pretty_dict_print(path_classifier_cfg, init_message="Path classifier configuration:")

## Loading data (images, centerlines, ground truths...)

In [ ]:
from math import ceil

max_dist = int(ceil(distances_hparams['max_dist']))

if max_dist is None:
    centerlines_dirname = "euclidean_all_centerlines"
else:
    centerlines_dirname = f"euclidean_lt_{max_dist}_centerlines"
print("Centerlines directory name:", centerlines_dirname)

In [ ]:
from path_neural_networks.data.image_centerline_dataset import ImageCenterlineDataset
from path_neural_networks.data.image_centerline_datamodule import ImageCenterlineDatamodule
import albumentations as A
from albumentations.pytorch import ToTensorV2

split_file_path=os.path.join(data_dir, "splits.json")
val_split_ratio = 0.2
use_foreground_pixels_only_for_normalization = True
data_seed = 42
split_seed = 42
shuffle_train = True

dataset = ImageCenterlineDataset(data_dir=data_dir, centerline_dirname=centerlines_dirname)
datamodule = ImageCenterlineDatamodule(dataset=dataset, 
                                        split_file_path=split_file_path,
                                        train_split_name=train_split,
                                        val_split_ratio=val_split_ratio,
                                        train_transforms=None,
                                        val_transforms=None,
                                        test_transforms=None,
                                        seed = split_seed,
                                        shuffle_train = shuffle_train)
datamodule.setup()

stats = dataset.get_dataset_stats(split_name=train_split, split_indices=datamodule.train_indices.tolist() + datamodule.val_indices.tolist())
if use_foreground_pixels_only_for_normalization:
    stats = stats['foreground']
else:
    stats = stats['full_image']
mean, std = stats['mean'], stats['std']
print("Dataset stats used for normalization:")
pretty_dict_print(stats)

### Adding data augmentation

In [ ]:
from path_neural_networks.data.augmentations import build_train_transform, build_val_transform

use_data_augmentation = True

train_transforms = build_train_transform(mean, std, use_data_augmentation=use_data_augmentation)
val_transforms = build_val_transform(mean, std)

datamodule = ImageCenterlineDatamodule(dataset=dataset,
                                        split_file_path=split_file_path,
                                        train_split_name=train_split,
                                        val_split_ratio=val_split_ratio,
                                        train_transforms=train_transforms,
                                        val_transforms=val_transforms,
                                        test_transforms=val_transforms,
                                        seed = split_seed,
                                        shuffle_train = shuffle_train)

### Initialize the loss function

In [ ]:
from path_neural_networks.models.losses import PathClassificationLoss, WeightedBCEWithLogitsLoss, BCEWithLogitsLoss

use_pos_weight_in_loss = True

loss_fn: PathClassificationLoss
if use_pos_weight_in_loss:
    classes_stats = dataset.get_dataset_classes_stats()
    classes_ratio = classes_stats['classes_ratio']
    loss_fn = WeightedBCEWithLogitsLoss(classes_ratio=classes_ratio)
else:
    loss_fn = BCEWithLogitsLoss()

## Initialize the model with all the previous components

In [ ]:
from path_neural_networks.models import ReducedPipelineLitModule
from path_neural_networks.utils.symmetry_enforcement import SymmetryEnforcementMode

learning_rate = 3e-4
metrics = ["accuracy", "auroc", "recall", "precision", "pr_auc"]
symmetry_enforcement_mode = SymmetryEnforcementMode.NONE

model = ReducedPipelineLitModule(
    features_generator=features_generator,
    path_sampler=path_sampler,
    path_encoder=path_encoder,
    path_classifier=path_classifier,
    edge_classification_loss_fn=loss_fn,
    metrics=metrics,
    lr=learning_rate,
    symmetry_enforcement_mode=symmetry_enforcement_mode
)

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_dir = dataset_choice.get_checkpoint_dir("main_model_training")
os.makedirs(checkpoint_dir, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        dirpath=checkpoint_dir,
        monitor="val_pr_auc",
        mode="max", 
        save_top_k=1, 
        filename="best-pr-auc-checkpoint-{epoch:02d}-{val_pr_auc:.4f}"
    ),
    EarlyStopping(
        monitor="val_pr_auc",
        patience=20,
        min_delta=1e-3,
        verbose=True,
        mode="max"
    ),
]

In [ ]:
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger

logger = CSVLogger(".")

torch.set_float32_matmul_precision("medium")
trainer = Trainer(accelerator='gpu', 
                  devices="auto",
                  num_nodes=1,
                  max_epochs=100,
                  precision="16-mixed",
                  detect_anomaly=False, 
                  callbacks=callbacks,
                  logger=logger,
                  gradient_clip_val=1.0,
                  gradient_clip_algorithm="norm",
                  val_check_interval=0.1
)
print(trainer)
print("Num GPUs:", trainer.num_devices)

In [ ]:
trainer.fit(model, datamodule=datamodule)